# Medical LLM Fine-Tune — Qwen2.5-0.5B + LoRA on MedQA

Fine-tunes **Qwen/Qwen2.5-0.5B-Instruct** on 1,000 real USMLE-style MedQA questions with LoRA. Runs on a free Colab T4 GPU in ~15 minutes. Repo: https://github.com/bitraj09/medical-llm-finetune

In [ ]:
# Medical LLM Fine-Tune — Qwen2.5-0.5B + LoRA on MedQA
!git clone https://github.com/bitraj09/medical-llm-finetune.git
%cd medical-llm-finetune

In [ ]:
# 1. Build the full training set from the official MedQA source (1,000 train + 200 eval)
!python data/prep_data.py
import json
rows = [json.loads(l) for l in open('data/medical_qa_train.jsonl')]
print(f'{len(rows)} training examples')
print(rows[0]['prompt'][:300])
print('->', rows[0]['completion'])

In [ ]:
# 2. LoRA fine-tune (1 epoch, ~15 min on a T4)
!python train.py --data data/medical_qa_train.jsonl --output outputs/lora-adapter

In [ ]:
# 3. Evaluate: base model vs fine-tuned adapter (200 held-out questions)
!python eval.py --data data/medical_qa_eval.jsonl
!python eval.py --data data/medical_qa_eval.jsonl --adapter outputs/lora-adapter

In [ ]:
# 4. Try the fine-tuned model on a fresh medical question
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct', torch_dtype=torch.bfloat16, device_map='auto')
model = PeftModel.from_pretrained(model, 'outputs/lora-adapter')
model.eval()

q = ('A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. '
     'Which of the following is the best treatment? A. Ampicillin B. Ceftriaxone C. Doxycycline D. Nitrofurantoin')
msgs = [{'role': 'user', 'content': q}]
text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tok(text, return_tensors='pt').to(model.device)
out = model.generate(**inputs, max_new_tokens=40, do_sample=False, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))